# Week 2 / Session 3 복습 랩 — 힌트 & 정답 포함

주제: **포트폴리오 비중 관리 — 분산인가 쏠림인가**

## 규칙
1. 각 문제는 **TODO 셀**에 코드를 작성하고, 바로 아래 **CHECK 셀**을 실행한다.
2. CHECK 셀이 `✅ PASS`를 출력해야 다음 문제로 넘어간다.
3. `❌ FAIL`이면 TODO 셀을 다시 고친다. 건너뛰지 말 것.
4. 막히면 **💡 1차 힌트** → **📖 2차 전체 설명** 순서로 펼친다.
5. 힌트 먼저 보지 말 것. 5분은 혼자 붙잡고 있어야 감이 붙는다.
6. 모든 CHECK가 `PASS`여야 과제 제출 가능.

---

**체크 목록** (9문제)
- Part 1 준비: Q1 import, Q2 샘플 CSV 생성
- Part 2 비중 계산: Q3 CSV 로드, Q4 평가금액, Q5 총자산 & 비중
- Part 3 구조 분석: Q6 섹터 groupby, Q7 HHI
- Part 4 AI 연결: Q8 요약 dict, Q9 OpenAI 호출

## 사전 설치

In [ ]:
!pip install pandas openai -q

---
# Part 1 — 준비

## Q1. 라이브러리 import

아래 2개를 **모두** import 하라:
- `pandas` as `pd`
- `OpenAI` (from `openai`)

In [ ]:
# TODO Q1
import pandas as pd
from openai import OpenAI

In [ ]:
# CHECK Q1
try:
    assert 'pd' in dir() and pd.__name__ == 'pandas', 'pandas as pd 누락'
    assert 'OpenAI' in dir(), 'openai OpenAI 누락'
    print('✅ PASS Q1')
except AssertionError as e:
    print(f'❌ FAIL Q1: {e}')

<details>
<summary><b>💡 Q1 1차 힌트</b> (클릭해서 펼치기)</summary>

- 단순 모듈: `import X as 별명`
- 모듈 안 특정 대상: `from X import Y`
- 2줄이면 충분하다.

</details>

<details>
<summary><b>📖 Q1 2차 전체 설명</b></summary>

```python
import pandas as pd
from openai import OpenAI
```

**왜 이렇게**
- `pandas as pd`: 관례. 커뮤니티 코드 거의 전부가 `pd`로 쓴다.
- `from openai import OpenAI`: 최신 openai SDK는 `OpenAI()` 클라이언트 인스턴스 패턴.

</details>

## Q2. 샘플 포트폴리오 CSV 생성

아래 스펙으로 `portfolio.csv`를 만들어라.

**컬럼**: `ticker, name, quantity, avg_price, current_price, sector`

**데이터**

| ticker | name | quantity | avg_price | current_price | sector |
|---|---|---|---|---|---|
| NVDA | NVIDIA | 10 | 800 | 950 | AI/반도체 |
| AMD  | AMD    | 15 | 120 | 140 | AI/반도체 |
| TSM  | TSMC   | 20 | 100 | 110 | 반도체 |
| AAPL | Apple  |  5 | 170 | 190 | IT/기기 |
| MSFT | Microsoft | 5 | 400 | 420 | IT/소프트웨어 |

`to_csv("portfolio.csv", index=False, encoding="utf-8-sig")`로 저장.

In [ ]:
# TODO Q2
sample_rows = [
    {"ticker": "NVDA", "name": "NVIDIA",    "quantity": 10, "avg_price": 800, "current_price": 950, "sector": "AI/반도체"},
    {"ticker": "AMD",  "name": "AMD",       "quantity": 15, "avg_price": 120, "current_price": 140, "sector": "AI/반도체"},
    {"ticker": "TSM",  "name": "TSMC",      "quantity": 20, "avg_price": 100, "current_price": 110, "sector": "반도체"},
    {"ticker": "AAPL", "name": "Apple",     "quantity":  5, "avg_price": 170, "current_price": 190, "sector": "IT/기기"},
    {"ticker": "MSFT", "name": "Microsoft", "quantity":  5, "avg_price": 400, "current_price": 420, "sector": "IT/소프트웨어"},
]
pd.DataFrame(sample_rows).to_csv("portfolio.csv", index=False, encoding="utf-8-sig")
print("✅ portfolio.csv 생성 완료")

In [ ]:
# CHECK Q2
import os
try:
    assert os.path.exists('portfolio.csv'), 'portfolio.csv 파일 없음'
    tmp = pd.read_csv('portfolio.csv')
    assert len(tmp) == 5, f'행 개수 오류 (기대 5, 실제 {len(tmp)})'
    for col in ['ticker','name','quantity','avg_price','current_price','sector']:
        assert col in tmp.columns, f'{col} 컬럼 누락'
    assert 'NVDA' in tmp['ticker'].values, 'NVDA 행 없음'
    print('✅ PASS Q2')
except Exception as e:
    print(f'❌ FAIL Q2: {e}')

<details>
<summary><b>💡 Q2 1차 힌트</b></summary>

- dict의 리스트를 만든 뒤 `pd.DataFrame(리스트)`에 넣으면 바로 표가 된다.
- 저장은 `df.to_csv("파일명", index=False, encoding="utf-8-sig")`.
- `index=False`가 없으면 0,1,2 행 번호가 파일에 같이 저장된다.

</details>

<details>
<summary><b>📖 Q2 2차 전체 설명</b></summary>

```python
sample_rows = [
    {"ticker": "NVDA", "name": "NVIDIA",    "quantity": 10, "avg_price": 800, "current_price": 950, "sector": "AI/반도체"},
    {"ticker": "AMD",  "name": "AMD",       "quantity": 15, "avg_price": 120, "current_price": 140, "sector": "AI/반도체"},
    {"ticker": "TSM",  "name": "TSMC",      "quantity": 20, "avg_price": 100, "current_price": 110, "sector": "반도체"},
    {"ticker": "AAPL", "name": "Apple",     "quantity":  5, "avg_price": 170, "current_price": 190, "sector": "IT/기기"},
    {"ticker": "MSFT", "name": "Microsoft", "quantity":  5, "avg_price": 400, "current_price": 420, "sector": "IT/소프트웨어"},
]
pd.DataFrame(sample_rows).to_csv("portfolio.csv", index=False, encoding="utf-8-sig")
```

**왜 utf-8-sig인가**
- 한글이 섞인 CSV를 엑셀에서 열면 깨지는 경우가 많다.
- `utf-8-sig`는 파일 맨 앞에 BOM을 붙여 엑셀이 "이건 UTF-8"이라고 인식하게 한다.

</details>

---
# Part 2 — 비중 계산

## Q3. CSV 로드

`portfolio.csv`를 읽어 `df`에 담아라. (`pd.read_csv` 사용)

In [ ]:
# TODO Q3
df = pd.read_csv("portfolio.csv")
df.head()

In [ ]:
# CHECK Q3
try:
    assert isinstance(df, pd.DataFrame), 'df가 DataFrame이 아님'
    assert len(df) == 5, f'행 개수 오류 (기대 5, 실제 {len(df)})'
    assert 'quantity' in df.columns and 'current_price' in df.columns, '필수 컬럼 누락'
    print('✅ PASS Q3')
except AssertionError as e:
    print(f'❌ FAIL Q3: {e}')

<details>
<summary><b>💡 Q3 1차 힌트</b></summary>

- 한 줄이다: `df = pd.read_csv("portfolio.csv")`
- 잘 읽혔는지 `df.head()`로 첫 5행만 찍어 보자.

</details>

<details>
<summary><b>📖 Q3 2차 전체 설명</b></summary>

```python
df = pd.read_csv("portfolio.csv")
df.head()
```

**포인트**
- `pd.read_csv`는 경로 + 인코딩 자동 추론으로 파일을 DataFrame으로 만든다.
- `head(n)`은 위에서 n행(기본 5)을 잘라 보여 준다. 전체 구조 확인용.
- 컬럼 목록은 `df.columns`, 행 수는 `len(df)` 또는 `df.shape`.

</details>

## Q4. 평가금액 컬럼 추가

`df`에 `value` 컬럼을 추가하라.

$$value = quantity \times current\_price$$

pandas는 벡터 연산을 지원해 모든 행이 한 번에 계산된다.

In [ ]:
# TODO Q4
df["value"] = df["quantity"] * df["current_price"]
df[["name", "quantity", "current_price", "value"]]

In [ ]:
# CHECK Q4
try:
    assert 'value' in df.columns, 'value 컬럼 없음'
    # NVDA: 10 * 950 = 9500
    nvda_value = df.loc[df['ticker']=='NVDA','value'].iloc[0]
    assert nvda_value == 9500, f'NVDA value 기대 9500, 실제 {nvda_value}'
    # MSFT: 5 * 420 = 2100
    msft_value = df.loc[df['ticker']=='MSFT','value'].iloc[0]
    assert msft_value == 2100, f'MSFT value 기대 2100, 실제 {msft_value}'
    print('✅ PASS Q4')
except AssertionError as e:
    print(f'❌ FAIL Q4: {e}')

<details>
<summary><b>💡 Q4 1차 힌트</b></summary>

- 새 컬럼 추가: `df["새컬럼"] = ...`
- 기존 컬럼끼리 곱: `df["A"] * df["B"]` — 반복문 필요 없음.

</details>

<details>
<summary><b>📖 Q4 2차 전체 설명</b></summary>

```python
df["value"] = df["quantity"] * df["current_price"]
```

**벡터 연산**
- `df["A"] * df["B"]`는 같은 인덱스 기준으로 행 단위로 곱한다.
- for 루프로 한 행씩 도는 것보다 10~100배 빠르다. 이게 pandas를 쓰는 이유다.
- 결과 Series를 `df["value"]`에 할당하면 새 컬럼이 붙는다.

**흔한 실수**
- `df["value"] = df["quantity"] * df["avg_price"]` — 평가금액은 **현재 가격(current_price)**으로 계산해야 한다. 평균 매입가가 아니다.

</details>

## Q5. 총 자산 & 비중 계산

- `total_value`: `value` 컬럼의 합
- `df["weight"]`: 각 종목이 전체에서 차지하는 비율 (value / total_value)

비중의 합은 반드시 **1.0**이 되어야 한다.

In [ ]:
# TODO Q5
total_value = df["value"].sum()
df["weight"] = df["value"] / total_value

print(f"총 자산: ${total_value:,.0f}")
print(f"비중 합: {df['weight'].sum():.4f}")
df[["name", "value", "weight"]]

In [ ]:
# CHECK Q5
try:
    assert 'total_value' in dir(), 'total_value 변수 없음'
    # 9500 + 2100 + 2200 + 950 + 2100 = 16850
    assert total_value == 16850, f'total_value 기대 16850, 실제 {total_value}'
    assert 'weight' in df.columns, 'weight 컬럼 없음'
    assert abs(df['weight'].sum() - 1.0) < 1e-6, f'비중 합 1.0 아님 ({df["weight"].sum()})'
    print('✅ PASS Q5')
except AssertionError as e:
    print(f'❌ FAIL Q5: {e}')

<details>
<summary><b>💡 Q5 1차 힌트</b></summary>

- 총합: `df["컬럼"].sum()`
- 비중: 각 행의 value를 총합으로 나눈다. `df["value"] / total_value`

</details>

<details>
<summary><b>📖 Q5 2차 전체 설명</b></summary>

```python
total_value = df["value"].sum()
df["weight"] = df["value"] / total_value
```

**포인트**
- `Series / 스칼라` 도 벡터 연산. 모든 행이 한 번에 나눠진다.
- 비중의 합이 1.0이 아니면 어딘가에서 값이 새는 중 — 이 검증(`sum() ≈ 1.0`)은 데이터 무결성 체크로 꼭 해 두면 좋다.
- 부동소수점 오차 때문에 `== 1.0`이 아니라 `abs(합 - 1.0) < 1e-6`로 비교한다.

</details>

---
# Part 3 — 구조 분석

## Q6. 섹터별 비중 집계 (groupby)

**Split → Apply → Combine**
- Split: `sector`별로 데이터 쪼개기
- Apply: 각 그룹의 `weight` 합 구하기
- Combine: 하나의 표로 합치기

결과는 `sector_weights` — 컬럼 `sector`, `weight`의 DataFrame으로, `weight` 내림차순 정렬.

In [ ]:
# TODO Q6
sector_weights = (
    df.groupby("sector")["weight"]
      .sum()
      .reset_index()
      .sort_values(by="weight", ascending=False)
)
sector_weights

In [ ]:
# CHECK Q6
try:
    assert isinstance(sector_weights, pd.DataFrame), 'sector_weights가 DataFrame이 아님'
    assert 'sector' in sector_weights.columns and 'weight' in sector_weights.columns, '컬럼 누락'
    assert len(sector_weights) == 4, f'섹터 개수 기대 4, 실제 {len(sector_weights)}'
    ai_weight = sector_weights.loc[sector_weights['sector']=='AI/반도체','weight'].iloc[0]
    assert abs(ai_weight - (9500+2100)/16850) < 1e-6, f'AI/반도체 비중 오류 ({ai_weight})'
    # 내림차순 검증
    w = sector_weights['weight'].tolist()
    assert w == sorted(w, reverse=True), '내림차순 정렬 아님'
    print('✅ PASS Q6')
except AssertionError as e:
    print(f'❌ FAIL Q6: {e}')

<details>
<summary><b>💡 Q6 1차 힌트</b></summary>

- `df.groupby("기준컬럼")["집계컬럼"].sum()` → sector를 인덱스로 하는 Series가 나온다.
- 다시 DataFrame으로 만들려면 `.reset_index()`.
- 정렬은 `.sort_values(by="weight", ascending=False)`.

</details>

<details>
<summary><b>📖 Q6 2차 전체 설명</b></summary>

```python
sector_weights = (
    df.groupby("sector")["weight"]
      .sum()
      .reset_index()
      .sort_values(by="weight", ascending=False)
)
```

**왜 groupby가 핵심인가**
- 종목을 5개 가지고 있어도 4개가 'AI/반도체'면 사실상 분산이 아니다.
- groupby는 이 "겉보기 분산"과 "실질 분산"의 차이를 드러낸다.

**체인 읽는 법**
1. `.groupby("sector")` — 섹터별 그룹 객체
2. `["weight"]` — 집계할 컬럼 선택
3. `.sum()` — 그룹별 합
4. `.reset_index()` — sector를 일반 컬럼으로 꺼냄
5. `.sort_values(...)` — 쏠린 섹터가 맨 위로 오게

</details>

## Q7. HHI (집중도 지표) 계산

$$HHI = \sum w_i^2$$

섹터 비중을 **제곱**해서 더한다. 제곱 때문에 큰 비중이 훨씬 부각되어 쏠림이 드러난다.

`hhi` 변수에 저장하고, 기준에 따라 `status` 문자열도 만들어라.

| HHI | status |
|---|---|
| < 0.15 | `"✅ 양호 (잘 분산됨)"` |
| 0.15 ~ 0.25 | `"⚖️ 주의 (집중됨)"` |
| > 0.25 | `"⚠️ 고위험 (매우 집중됨)"` |

In [ ]:
# TODO Q7
hhi = (sector_weights["weight"] ** 2).sum()

if hhi > 0.25:
    status = "⚠️ 고위험 (매우 집중됨)"
elif hhi > 0.15:
    status = "⚖️ 주의 (집중됨)"
else:
    status = "✅ 양호 (잘 분산됨)"

print(f"HHI: {hhi:.3f}  →  {status}")

In [ ]:
# CHECK Q7
try:
    assert 'hhi' in dir(), 'hhi 변수 없음'
    expected = (sector_weights['weight']**2).sum()
    assert abs(hhi - expected) < 1e-6, f'HHI 값 오류 (기대 {expected:.4f}, 실제 {hhi:.4f})'
    assert 0 < hhi < 1, f'HHI 범위 이상 ({hhi})'
    assert 'status' in dir() and isinstance(status, str), 'status 문자열 없음'
    # 본 샘플은 HHI ≈ 0.354 → 고위험
    assert '고위험' in status, f'status 판정 오류 (hhi={hhi:.3f}, status={status})'
    print(f'✅ PASS Q7 (HHI={hhi:.3f})')
except AssertionError as e:
    print(f'❌ FAIL Q7: {e}')

<details>
<summary><b>💡 Q7 1차 힌트</b></summary>

- 제곱: `** 2`. `sector_weights["weight"] ** 2`는 각 원소가 제곱된 Series.
- 합: `.sum()`.
- status는 `if/elif/else` 3분기.

</details>

<details>
<summary><b>📖 Q7 2차 전체 설명</b></summary>

```python
hhi = (sector_weights["weight"] ** 2).sum()

if hhi > 0.25:
    status = "⚠️ 고위험 (매우 집중됨)"
elif hhi > 0.15:
    status = "⚖️ 주의 (집중됨)"
else:
    status = "✅ 양호 (잘 분산됨)"
```

**왜 제곱인가**
- 단순 합은 항상 1이어서 쓸모가 없다.
- 제곱하면 큰 값이 훨씬 커지고 작은 값은 거의 무시된다. → 쏠림에 민감한 지표가 된다.
- 예: 5개 섹터 균등 20%씩 → HHI=0.2. 한 섹터에 100% 몰림 → HHI=1.0.

**조건문 순서 주의**
- `elif hhi > 0.15`는 이미 `hhi <= 0.25`인 상태에서 검사되므로 `0.15 < hhi <= 0.25` 구간을 정확히 잡는다.

</details>

---
# Part 4 — AI 연결

## Q8. AI 비평용 요약 dict

AI 프롬프트에 꽂아 쓰기 좋은 dict `analysis_summary`를 만들어라.

구조:
```python
{
    "sector_data": [ {"sector": ..., "weight": ...}, ... ],
    "hhi_score": 0.354
}
```

힌트: `to_dict(orient="records")`, `round(hhi, 3)`.

In [ ]:
# TODO Q8
analysis_summary = {
    "sector_data": sector_weights.to_dict(orient="records"),
    "hhi_score": round(hhi, 3),
}
analysis_summary

In [ ]:
# CHECK Q8
try:
    assert isinstance(analysis_summary, dict), 'dict가 아님'
    assert 'sector_data' in analysis_summary and 'hhi_score' in analysis_summary, '키 누락'
    sd = analysis_summary['sector_data']
    assert isinstance(sd, list) and len(sd) == 4, 'sector_data 리스트 길이 이상'
    assert all('sector' in r and 'weight' in r for r in sd), 'sector_data 원소 형식 오류'
    assert abs(analysis_summary['hhi_score'] - round(hhi,3)) < 1e-9, 'hhi_score 값 오류'
    print('✅ PASS Q8')
except AssertionError as e:
    print(f'❌ FAIL Q8: {e}')

<details>
<summary><b>💡 Q8 1차 힌트</b></summary>

- DataFrame → 리스트의 dict: `df.to_dict(orient="records")`
- 예: `[{"sector":"AI/반도체","weight":0.688}, ...]`
- 반올림: `round(값, 자리수)`

</details>

<details>
<summary><b>📖 Q8 2차 전체 설명</b></summary>

```python
analysis_summary = {
    "sector_data": sector_weights.to_dict(orient="records"),
    "hhi_score": round(hhi, 3),
}
```

**orient 옵션**
- `"records"`: 행마다 dict. LLM이 제일 읽기 편한 형태.
- `"dict"`: 컬럼마다 dict (중첩). LLM엔 읽기 힘듦.
- `"list"`: 컬럼마다 값 리스트. 행 관계가 분리됨.

**왜 반올림**
- `0.3541234567` 같은 값을 프롬프트에 그대로 넣으면 토큰만 낭비.
- 3자리 정도면 해석용으로 충분하다.

</details>

]Y]yh## Q9. OpenAI 호출 — 퀀트 비평

아래 요구대로 AI 비평을 `ai_critique`에 저장하라.

- 모델: `gpt-4o`
- system: "월스트리트 15년차 헤지펀드 퀀트 애널리스트" 역할, 아래 4줄 포맷만 답하게 지시
  - `📊 현재 구조:` / `🎯 핵심 리스크:` / `💡 리밸런싱 제안:` / `⚠️ 모니터링 포인트:`
- user: `analysis_summary`의 섹터 비중과 HHI를 문자열로 넣어 비평 요청
- `temperature=0.3`, `max_tokens=500`

`OPENAI_API_KEY`는 환경변수(`os.environ`) 또는 Colab `userdata`에서 가져온다.

In [ ]:
# TODO Q9
import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
# Colab: from google.colab import userdata; OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """
너는 월스트리트 15년차 헤지펀드 퀀트 애널리스트다.
사용자의 포트폴리오 섹터별 비중과 HHI 지수를 받아 리스크를 비평한다.
반드시 아래 형식으로만 답하라.

📊 현재 구조: (한 줄 요약)
🎯 핵심 리스크: (쏠림 섹터 또는 HHI 해석)
💡 리밸런싱 제안: (어떤 섹터 비중을 어떻게 조절할지 구체적으로)
⚠️ 모니터링 포인트: (앞으로 지켜볼 지표 한 줄)
"""

user_prompt = (
    f"내 포트폴리오의 섹터별 비중은 {analysis_summary['sector_data']}이고, "
    f"HHI 지수는 {analysis_summary['hhi_score']}다. "
    "이 포트가 특정 섹터에 쏠려 있는지 퀀트 관점에서 비평하고, "
    "리스크를 낮추기 위한 제안을 하라."
)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ],
    max_tokens=500,
    temperature=0.3,
)
ai_critique = response.choices[0].message.content
print("🤖 AI 퀀트 비평")
print("="*40)
print(ai_critique)

In [ ]:
# CHECK Q9
try:
    assert 'client' in dir() and client.__class__.__name__ == 'OpenAI', 'client 없음'
    assert isinstance(ai_critique, str), 'ai_critique 문자열 아님'
    assert len(ai_critique) > 50, '응답이 너무 짧음'
    print(f'✅ PASS Q9 (길이 {len(ai_critique)})')
except AssertionError as e:
    print(f'❌ FAIL Q9: {e}')

<details>
<summary><b>💡 Q9 1차 힌트</b></summary>

- 환경에 따라 키 꺼내는 방식이 다르다:
  - Colab: `from google.colab import userdata` 후 `userdata.get('OPENAI_API_KEY')`
  - 로컬: `os.environ.get("OPENAI_API_KEY")`
- `client = OpenAI(api_key=...)` → `client.chat.completions.create(model=, messages=, ...)`
- messages는 `[{system}, {user}]` 2개 딕셔너리 리스트.

</details>

<details>
<summary><b>📖 Q9 2차 전체 설명</b></summary>

```python
import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ],
    max_tokens=500,
    temperature=0.3,
)
ai_critique = response.choices[0].message.content
```

**프롬프트 설계 포인트**
- **system**: 모델에게 "누구"인지, "어떻게" 답할지 지정. 여기선 역할(퀀트)과 출력 포맷(4줄)을 못 박는다.
- **user**: 실제 데이터와 요청. 숫자는 계산해서 넣어 주고, 모델은 해석만 하게 한다.
- **temperature=0.3**: 분석은 낮게. 0에 가까울수록 결정적이고 보수적인 답.

**계산은 코드, 해석은 LLM** — 이 분리가 핵심이다.

</details>

---
# 최종 점검

아래 셀을 실행해 9문제의 PASS 개수를 확인한다.

In [ ]:
# FINAL SCOREBOARD
import os
checks = {
    'Q1': lambda: 'pd' in dir() and 'OpenAI' in dir(),
    'Q2': lambda: os.path.exists('portfolio.csv'),
    'Q3': lambda: isinstance(df, pd.DataFrame) and len(df)==5,
    'Q4': lambda: 'value' in df.columns and df.loc[df['ticker']=='NVDA','value'].iloc[0]==9500,
    'Q5': lambda: total_value==16850 and abs(df['weight'].sum()-1.0)<1e-6,
    'Q6': lambda: isinstance(sector_weights, pd.DataFrame) and len(sector_weights)==4,
    'Q7': lambda: isinstance(hhi, float) and 0<hhi<1 and isinstance(status, str),
    'Q8': lambda: isinstance(analysis_summary, dict) and 'sector_data' in analysis_summary,
    'Q9': lambda: isinstance(ai_critique, str) and len(ai_critique)>50,
}
passed = 0
for q, fn in checks.items():
    try:
        ok = fn()
    except Exception:
        ok = False
    mark = '✅' if ok else '❌'
    print(f'{mark} {q}')
    passed += 1 if ok else 0
print(f'\n=== {passed}/9 PASS ===')
print('합격' if passed == 9 else '불합격 — FAIL 항목 재시도')